# Week 11: Greedy Algorithms & Minimum Spanning Trees

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Union-Find DSU, Kruskal MST, Prim MST, interval scheduling, and fractional knapsack.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-11/labs/W11_Greedy_and_MST_Kruskal_Prim.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Disjoint Set Union, Kruskal's MST & Prim's MST

Both Kruskal (global sorted edges + Union-Find) and Prim (local cut expansion + min-heap) obey the MST cut property and produce identical total tree weights.

In [1]:
import heapq
import random


class DisjointSet:
    def __init__(self, items):
        self.p = {x: x for x in items}
        self.r = {x: 0 for x in items}

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.r[ra] < self.r[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]:
            self.r[ra] += 1
        return True


def kruskal_mst(verts, edges):
    dsu = DisjointSet(verts)
    mst, total = [], 0.0
    for u, v, w in sorted(edges, key=lambda e: e[2]):
        if dsu.union(u, v):
            mst.append((u, v, w))
            total += w
    return total, mst


def prim_mst(adj, start):
    seen = {start}
    pq = [(w, 0, start, v) for v, w in adj[start]]
    heapq.heapify(pq)
    mst, total, cid = [], 0.0, 0
    while pq and len(seen) < len(adj):
        w, _, u, v = heapq.heappop(pq)
        if v in seen:
            continue
        seen.add(v)
        mst.append((u, v, w))
        total += w
        for nxt, nw in adj[v]:
            if nxt not in seen:
                cid += 1
                heapq.heappush(pq, (nw, cid, v, nxt))
    return total, mst


verts = ["StoneTown", "Bububu", "Mbweni", "Fumba", "Chwaka", "Kizimkazi"]
edges = [
    ("StoneTown", "Bububu", 9), ("StoneTown", "Mbweni", 6), ("Mbweni", "Fumba", 14),
    ("StoneTown", "Chwaka", 28), ("Bububu", "Chwaka", 22), ("Fumba", "Kizimkazi", 41),
    ("Chwaka", "Kizimkazi", 34), ("Mbweni", "Chwaka", 26),
]
adj = {v: [] for v in verts}
for u, v, w in edges:
    adj[u].append((v, w)); adj[v].append((u, w))

kw, kmst = kruskal_mst(verts, edges)
pw, _ = prim_mst(adj, "StoneTown")
assert kw == pw
print(f"Zanzibar fiber backbone MST total = {kw:.0f} km, edges = {kmst}")


Zanzibar fiber backbone MST total = 85 km, edges = [('StoneTown', 'Mbweni', 6), ('StoneTown', 'Bububu', 9), ('Mbweni', 'Fumba', 14), ('Bububu', 'Chwaka', 22), ('Chwaka', 'Kizimkazi', 34)]


## 2. Greedy Exchange Arguments: Interval Scheduling & Fractional Knapsack

Sorting intervals by earliest finish time and items by value-to-weight density achieves global optimality via greedy exchange proofs.

In [2]:
def interval_scheduling(intervals):
    chosen, end_time = [], -float("inf")
    for s, e in sorted(intervals, key=lambda iv: (iv[1], iv[0])):
        if s >= end_time:
            chosen.append((s, e))
            end_time = e
    return chosen


def fractional_knapsack(cap, items):
    total = 0.0
    for val, wt in sorted(items, key=lambda x: x[0] / x[1], reverse=True):
        take = min(wt, cap)
        total += val * (take / wt)
        cap -= take
        if cap <= 0:
            break
    return total


picked = interval_scheduling([(9, 10), (9, 12), (10, 11), (11, 13), (12, 14)])
assert len(picked) == 3
fk = fractional_knapsack(50, [(60, 10), (100, 20), (120, 30)])
assert fk == 240.0
print("Scheduled lectures:", picked, "| Fractional knapsack:", fk)


Scheduled lectures: [(9, 10), (10, 11), (11, 13)] | Fractional knapsack: 240.0
